In [ ]:
def main(datasources, start_date, end_date):
    """BigAlpha 2026 single-factor submission: factor_xover_late_reversal_3_low_imbalance_flip_36ccf047_g37_mut_g45

    Research direction: Downside-risk-normalized late reversal x low L5 sign-flip rate.
    Uses only historical CSI1000 constituents and minute bars ending strictly
    before 15:00.  Query end is never expanded; warm-up is cropped before return.
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]
    eps = 1e-9
    evaluation_start = pd.to_datetime(start_date)
    evaluation_end = pd.to_datetime(end_date)
    query_start = evaluation_start - pd.Timedelta(days=150)

    sql = f"""
WITH base AS (
    SELECT
        b.date AS minute_ts,
        b.instrument,
        CAST(b.date AS DATE) AS trading_day,
        b.close,
        COALESCE(b.bid_volume1, 0) AS bid_volume1,
        COALESCE(b.bid_volume2, 0) AS bid_volume2,
        COALESCE(b.bid_volume3, 0) AS bid_volume3,
        COALESCE(b.bid_volume4, 0) AS bid_volume4,
        COALESCE(b.bid_volume5, 0) AS bid_volume5,
        COALESCE(b.ask_volume1, 0) AS ask_volume1,
        COALESCE(b.ask_volume2, 0) AS ask_volume2,
        COALESCE(b.ask_volume3, 0) AS ask_volume3,
        COALESCE(b.ask_volume4, 0) AS ask_volume4,
        COALESCE(b.ask_volume5, 0) AS ask_volume5
    FROM {bar1m} AS b
    INNER JOIN bigalpha_2026_instruments AS p
        ON b.instrument = p.instrument
        AND CAST(b.date AS DATE) = CAST(p.date AS DATE)
    WHERE strftime(b.date, '%H:%M:%S') < '15:00:00'
),
row_values AS (
    SELECT
        *,
        ((bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5) - (ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5)) / NULLIF((bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5) + (ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5), 0) AS imbalance_l5_row
    FROM base
),
sequenced AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY instrument, trading_day ORDER BY minute_ts) AS forward_rank,
        ROW_NUMBER() OVER (PARTITION BY instrument, trading_day ORDER BY minute_ts DESC) AS reverse_rank,
        close / NULLIF(LAG(close) OVER (PARTITION BY instrument, trading_day ORDER BY minute_ts), 0) - 1 AS minute_return
    FROM row_values
),
path_values AS (
    SELECT
        *,
        CASE WHEN LAG(SIGN(imbalance_l5_row)) OVER (PARTITION BY instrument, trading_day ORDER BY minute_ts) IS NULL THEN 0.0 WHEN SIGN(imbalance_l5_row) <> LAG(SIGN(imbalance_l5_row)) OVER (PARTITION BY instrument, trading_day ORDER BY minute_ts) THEN 1.0 ELSE 0.0 END AS imbalance_l5_flip
    FROM sequenced
),
daily_raw AS (
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        COUNT(*) AS n_minutes,
        ARG_MAX(close, minute_ts) / NULLIF(MAX(CASE WHEN reverse_rank = 31 THEN close END), 0) - 1 AS late_ret_30,
        AVG(imbalance_l5_flip) AS imbalance_l5_flip_rate,
        SUM(CASE WHEN minute_return < 0 THEN POWER(minute_return, 2) ELSE 0 END) AS downside_rv
    FROM path_values
    GROUP BY trading_day, instrument
),
daily AS (
    SELECT
        date,
    instrument,
    downside_rv,
    imbalance_l5_flip_rate,
    late_ret_30
    FROM daily_raw
)
SELECT *
FROM daily
ORDER BY instrument, date
    """

    daily_parts = []
    chunk_start = query_start.normalize()
    while chunk_start <= evaluation_end:
        next_chunk_start = chunk_start + pd.Timedelta(days=5)
        chunk_end = min(next_chunk_start - pd.Timedelta(seconds=1), evaluation_end)
        part = dai.query(
            sql,
            filters={
                "date": [
                    chunk_start.strftime("%Y-%m-%d %H:%M:%S"),
                    chunk_end.strftime("%Y-%m-%d %H:%M:%S"),
                ]
            },
            compression=True,
        ).df()
        if not part.empty:
            daily_parts.append(part)
        chunk_start = next_chunk_start

    if daily_parts:
        daily = pd.concat(daily_parts, ignore_index=True)
    else:
        daily = pd.DataFrame(columns=["date", "instrument"] + ['downside_rv', 'imbalance_l5_flip_rate', 'late_ret_30'])

    daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
    daily["instrument"] = daily["instrument"].astype(str)
    daily = daily.drop_duplicates(["date", "instrument"], keep="last")
    for column in ['downside_rv', 'imbalance_l5_flip_rate', 'late_ret_30']:
        daily[column] = pd.to_numeric(daily[column], errors="coerce")
    daily = daily.sort_values(["instrument", "date"]).reset_index(drop=True)

    def rolling_mean(column, window, min_periods):
        return daily.groupby("instrument", sort=False)[column].transform(
            lambda values: values.rolling(window, min_periods=min_periods).mean()
        )

    def rolling_std(column, window, min_periods):
        return daily.groupby("instrument", sort=False)[column].transform(
            lambda values: values.rolling(window, min_periods=min_periods).std()
        )

    daily["left_tmp_late_ma"] = rolling_mean("late_ret_30", 5, 3)
    daily["left_tmp_ds_rv"] = rolling_mean("downside_rv", 20, 10)
    daily["left_raw"] = -daily["left_tmp_late_ma"] / (daily["left_tmp_ds_rv"] + 1e-6)
    daily["right_raw"] = -rolling_mean("imbalance_l5_flip_rate", 5, 3)
    daily["left_mean"] = rolling_mean("left_raw", 30, 10)
    daily["left_std"] = rolling_std("left_raw", 30, 10)
    daily["right_mean"] = rolling_mean("right_raw", 30, 10)
    daily["right_std"] = rolling_std("right_raw", 30, 10)
    daily["factor"] = 0.5 * (
        (daily["left_raw"] - daily["left_mean"]) / (daily["left_std"] + eps)
        + (daily["right_raw"] - daily["right_mean"]) / (daily["right_std"] + eps)
    )

    daily = daily[
        (daily["date"] >= evaluation_start.normalize())
        & (daily["date"] <= evaluation_end)
    ].copy()
    result = daily[["date", "instrument", "factor"]].copy()
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan)

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"], errors="coerce")
    stk_pool["instrument"] = stk_pool["instrument"].astype(str)
    stk_pool = stk_pool.drop_duplicates(["date", "instrument"])
    result = pd.merge(
        stk_pool[["date", "instrument"]],
        result,
        how="left",
        on=["date", "instrument"],
    )

    # Research coverage is about 90%; this only fills the remaining cross-section.
    # If an entire day is empty, the final zero is explicit and deterministic.
    result["factor"] = result.groupby("date")["factor"].transform(
        lambda values: values.fillna(values.median())
    )
    result["factor"] = result["factor"].fillna(0.0)
    return result[["date", "instrument", "factor"]].copy()
